# Conventional Approach — XGBoost with Optuna

Loads the train and test sets written by `src/split_data.py`, tunes an XGBoost model with Optuna (a Bayesian TPE search) using **time-ordered cross-validation inside the training set**, fits the best parameters on the whole training set, and evaluates **once** on the held-out test set.

All logic is in `src/xgboost_tuning.py`; this notebook calls it and shows the results. Run `python src/split_data.py` first if `train.csv` and `test.csv` are missing.

In [2]:
import sys; print(sys.executable)

/usr/local/bin/python3.13


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from xgboost_tuning import (N_TRIALS, RANDOM_SEED, RESULTS_FILE, THRESHOLD, TRAIN_FRACTIONS,
                            check_folds, evaluate, fit_final, load_split, save_results,
                            time_ordered_folds, tune)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

## 1. Load train and test

`first_date` and `churn` are separated from the features. `first_date` is kept only to order the training rows for the CV folds; it is never fed to the model.

In [2]:
X_train, y_train, dates_train, X_test, y_test = load_split()

assert list(X_train.columns) == list(X_test.columns), 'train/test features differ'
assert 'first_date' not in X_train.columns and 'churn' not in X_train.columns

print(f'Train : {X_train.shape}  churn rate {y_train.mean():.3f}  '
      f'({dates_train.min().date()} -> {dates_train.max().date()})')
print(f'Test  : {X_test.shape}  churn rate {y_test.mean():.3f}')
nan_cols = X_train.isna().sum()
print(f'\nFeatures with missing values in train: {(nan_cols > 0).sum()} '
      f'(XGBoost handles NaN natively)')
print(f'Non-numeric feature columns: {list(X_train.select_dtypes(exclude="number").columns)}')

Train : (4035, 173)  churn rate 0.447  (2009-12-01 -> 2010-11-30)
Test  : (1009, 173)  churn rate 0.494

Features with missing values in train: 40 (XGBoost handles NaN natively)
Non-numeric feature columns: []


In [5]:
[i for i in (X_train.columns)]

['year',
 'month',
 'day_of_month',
 'hour',
 'n_lines',
 'n_distinct_products',
 'total_quantity',
 'total_spend',
 'avg_unit_price',
 'weekday_0',
 'weekday_1',
 'weekday_2',
 'weekday_4',
 'weekday_5',
 'weekday_6',
 'time_of_day_evening',
 'time_of_day_morning',
 'country_Australia',
 'country_Austria',
 'country_Belgium',
 'country_Channel_Islands',
 'country_France',
 'country_Germany',
 'country_Italy',
 'country_Netherlands',
 'country_Other',
 'country_Portugal',
 'country_Spain',
 'country_Sweden',
 'country_Switzerland',
 'prior_transactions_min',
 'prior_transactions_max',
 'prior_transactions_mean',
 'prior_transactions_median',
 'prior_transactions_sum',
 'prior_units_min',
 'prior_units_max',
 'prior_units_mean',
 'prior_units_median',
 'prior_units_sum',
 'prior_median_daily_transactions_min',
 'prior_median_daily_transactions_max',
 'prior_median_daily_transactions_mean',
 'prior_median_daily_transactions_median',
 'prior_median_daily_transactions_sum',
 'prior_median_

## 2. Time-ordered cross-validation folds

The training customers are ordered by `first_date`. Fold `0.6` trains on the earliest 60% and validates on the remaining, latest 40%; `0.7` trains on the earliest 70% and validates on the latest 30%; likewise `0.8` and `0.9`. Validation windows overlap and all end at the last training date, but every validation customer is later than every training customer in the same fold.

In [3]:
folds = time_ordered_folds(dates_train)
check_folds(dates_train, folds)

dates = dates_train.reset_index(drop=True)
rows = []
for fraction, (train_idx, val_idx) in zip(TRAIN_FRACTIONS, folds):
    rows.append({
        'fold (train share)': fraction,
        'train rows': len(train_idx),
        'validation rows': len(val_idx),
        'train: last date': dates.iloc[train_idx].max().date(),
        'validation: first date': dates.iloc[val_idx].min().date(),
        'validation churn rate': y_train.reset_index(drop=True).iloc[val_idx].mean(),
    })
pd.DataFrame(rows).round(3)

,fold (train share),train rows,validation rows,train: last date,validation: first date,validation churn rate
0,0.6,2421,1614,2010-05-19,2010-05-19,0.495
1,0.7,2824,1211,2010-07-06,2010-07-06,0.491
2,0.8,3227,808,2010-09-19,2010-09-19,0.512
3,0.9,3632,403,2010-10-24,2010-10-24,0.566


## 3. Hyperparameter tuning with Optuna

Optuna's TPE sampler (seeded with `RANDOM_SEED = 42`) runs `N_TRIALS = 100` trials. Each trial fits one model per fold and its score is the **mean validation AUC-ROC over the 4 folds**. The search space is in `suggest_params` in `src/xgboost_tuning.py`.

In [4]:
study = tune(X_train, y_train, folds, n_trials=N_TRIALS, seed=RANDOM_SEED, show_progress_bar=True)

print(f'\nBest mean CV AUC-ROC : {study.best_value:.4f}  (trial {study.best_trial.number})')
print('Per-fold AUC-ROC     :', dict(zip(TRAIN_FRACTIONS,
      [round(a, 4) for a in study.best_trial.user_attrs["fold_aucs"]])))
print('Best parameters      :')
for name, value in study.best_params.items():
    print(f'  {name:<18}: {value:.6g}' if isinstance(value, float) else f'  {name:<18}: {value}')

  0%|          | 0/100 [00:00<?, ?it/s]


Best mean CV AUC-ROC : 0.6082  (trial 55)
Per-fold AUC-ROC     : {0.6: 0.5961, 0.7: 0.6024, 0.8: 0.6209, 0.9: 0.6132}
Best parameters      :
  n_estimators      : 333
  max_depth         : 3
  learning_rate     : 0.0132327
  subsample         : 0.729944
  colsample_bytree  : 0.864424
  min_child_weight  : 4
  reg_alpha         : 2.40014
  reg_lambda        : 0.000916311


## 4. Fit the final model on the whole training set

In [5]:
final_model = fit_final(study.best_params, X_train, y_train)
print(f'Final model fitted on {len(X_train):,} training customers, {X_train.shape[1]} features')

Final model fitted on 4,035 training customers, 173 features


## 5. Evaluate on the held-out test set

Threshold-based metrics use a probability threshold of 0.5; AUC-ROC is threshold-free. The test set is used only here.

In [6]:
metrics = evaluate(final_model, X_test, y_test)

summary = pd.Series({k: metrics[k] for k in ('accuracy', 'precision', 'recall', 'f1', 'auc_roc')},
                    name='test set').round(4)
print(f'=== Test set results (threshold {THRESHOLD}, churn = positive class) ===')
print(summary.to_string())

print('\nConfusion matrix')
display(pd.DataFrame([[metrics['tn'], metrics['fp']], [metrics['fn'], metrics['tp']]],
                     index=pd.Index(['actual retained', 'actual churned'], name=''),
                     columns=['predicted retained', 'predicted churned']))

print(f'Mean CV AUC-ROC {study.best_value:.4f} vs test AUC-ROC {metrics["auc_roc"]:.4f}')

=== Test set results (threshold 0.5, churn = positive class) ===
accuracy     0.5956
precision    0.5904
recall       0.5904
f1           0.5904
auc_roc      0.6113

Confusion matrix


,predicted retained,predicted churned
,,
actual retained,307,204
actual churned,204,294


Mean CV AUC-ROC 0.6082 vs test AUC-ROC 0.6113


## 6. Save results

Metrics, best parameters and CV details go to `results/conventional_xgboost.json`.

In [7]:
out_path = save_results(study, metrics, X_train, X_test, folds)
print(f'Results saved to {out_path.relative_to(out_path.parents[1])}')

Results saved to results/conventional_xgboost.json
